# ThirdEye – Thermal Military Vehicle Detection, Tracking & Counting
Pipeline: CLAHE preprocessing → YOLO11 → (SAHI for small objects) → ByteTrack → **track-level class voting** → unique-target counting → HUD video.

Run top to bottom on Colab (GPU). Cells 1-3 reuse your existing merged dataset at `/content/unified_military_thermal`.

In [1]:
!pip install -q ultralytics sahi lap pandas
import os, glob, shutil, random, cv2, yaml, numpy as np, pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from ultralytics import YOLO

DATA_DIR = Path('/content/unified_military_thermal')   # your merged dataset
CLAHE_DIR = Path('/content/unified_clahe')              # CLAHE-processed copy
MODEL_BASE = 'yolo11s.pt'                               # try yolo11m.pt if GPU memory allows
IMGSZ = 800                                             # compromise: better for small objects, fits 8GB + time budget
EPOCHS = 40                                              # ~25 min per run on RTX 5050 (est.)
BATCH = 8                                               # 8GB VRAM safe
RUN_SCALING = True                                      # set False to save ~15 min
data_yaml = yaml.safe_load(open(DATA_DIR/'data.yaml'))
CLASS_NAMES = data_yaml['names'] if isinstance(data_yaml['names'], list) else list(data_yaml['names'].values())
print(CLASS_NAMES)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


['Soldier', 'Tank', 'APC', 'Military_Truck', 'Mobile_Air_Defense']


In [2]:
! pip install -q ultralytics sahi lap panda


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Thermal preprocessing (CLAHE)

In [3]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess(frame):
    """Grayscale -> CLAHE -> back to 3 channels. Used for training AND inference."""
    g = frame if frame.ndim == 2 else cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(_clahe.apply(g), cv2.COLOR_GRAY2BGR)

def build_clahe_dataset(src, dst):
    if dst.exists(): shutil.rmtree(dst)
    for split in ['train', 'val', 'test']:
        (dst/'images'/split).mkdir(parents=True); (dst/'labels'/split).mkdir(parents=True)
        for p in glob.glob(str(src/'images'/split/'*.*')):
            img = cv2.imread(p)
            if img is None: continue
            cv2.imwrite(str(dst/'images'/split/Path(p).name), preprocess(img))
        for l in glob.glob(str(src/'labels'/split/'*.txt')):
            shutil.copy(l, dst/'labels'/split/Path(l).name)
    cfg = dict(data_yaml); cfg['path'] = str(dst)
    cfg.update(train='images/train', val='images/val', test='images/test')
    yaml.safe_dump(cfg, open(dst/'data.yaml', 'w'))

build_clahe_dataset(DATA_DIR, CLAHE_DIR)

# make sure the raw dataset yaml has absolute path + test split too
cfg = dict(data_yaml); cfg['path'] = str(DATA_DIR)
cfg.update(train='images/train', val='images/val', test='images/test')
yaml.safe_dump(cfg, open(DATA_DIR/'data.yaml', 'w'))

## 2. Train: baseline (raw) vs CLAHE – this is your ablation

In [4]:
# Thermal-appropriate augmentation: no hue/saturation (meaningless in IR), flips + rotation + scale
AUG = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, flipud=0.5, fliplr=0.5, degrees=10,
           scale=0.5, mosaic=1.0, close_mosaic=10)

def train(name, yaml_path, **extra):
    m = YOLO(MODEL_BASE)
    results = m.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=0,      # <-- FIX: Set to 0 on Windows to prevent worker crashes
        cache=False,    # <-- FIX: Set to False to prevent RAM exhaustion
        patience=15,
        optimizer='AdamW',
        lr0=0.001,
        device=0,
        project='ThirdEye',
        name=name,
        exist_ok=True,
        **AUG,
        **extra
    )
    # Ultralytics directly tracks the best weights path:
    return Path(m.trainer.best)

# Reuse existing 'raw' model weights if already trained:
raw_best_local = Path(r'C:\Users\Viraj\runs\detect\ThirdEye\raw\weights\best.pt')
if raw_best_local.exists():
    print(f"Found existing raw weights at {raw_best_local}, skipping re-train!")
    w_raw = raw_best_local
else:
    w_raw = train('raw', DATA_DIR/'data.yaml')

# Train CLAHE
w_clahe = train('clahe', CLAHE_DIR/'data.yaml')


Found existing raw weights at C:\Users\Viraj\runs\detect\ThirdEye\raw\weights\best.pt, skipping re-train!
New https://pypi.org/project/ultralytics/8.4.187 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.75  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=\content\unified_clahe\data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=

## 3. Ablation table (test split – report these numbers)

In [5]:
def evaluate(weights, yaml_path, tag, **kw):
    m = YOLO(str(weights))
    r = m.val(data=str(yaml_path), split='test', imgsz=IMGSZ, verbose=False, **kw)
    return dict(experiment=tag, precision=r.box.mp, recall=r.box.mr, mAP50=r.box.map50, mAP50_95=r.box.map)

rows = [
    evaluate(w_raw,   DATA_DIR/'data.yaml',   'YOLO11 raw'),
    evaluate(w_clahe, CLAHE_DIR/'data.yaml',  'YOLO11 + CLAHE'),
    evaluate(w_clahe, CLAHE_DIR/'data.yaml',  'YOLO11 + CLAHE + TTA', augment=True),
]
ablation = pd.DataFrame(rows).round(3)
display(ablation)
ablation.to_csv('ablation_results.csv', index=False)

Ultralytics 8.4.75  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 81.732.0 MB/s, size: 51.1 KB)
val: Scanning C:\content\unified_military_thermal\labels\test... 63 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 63/63 1.0Kit/s 0.1s
val: New cache created: C:\content\unified_military_thermal\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.5s/it 5.9s0.8s1s
                   all         63        193      0.663      0.309      0.338      0.128
Speed: 6.6ms preprocess, 14.4ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to C:\Users\Viraj\runs\detect\val-3
Ultralytics 8.4.75  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 paramet

,experiment,precision,recall,mAP50,mAP50_95
0,YOLO11 raw,0.663,0.309,0.338,0.128
1,YOLO11 + CLAHE,0.291,0.397,0.303,0.125
2,YOLO11 + CLAHE + TTA,0.368,0.402,0.325,0.135


## 4. SAHI – sliced inference for small objects (soldiers, distant vehicles)

In [6]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

best = str(w_clahe)
sahi_model = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=best,
                                                confidence_threshold=0.25, device='cuda:0')

def detect_sahi(img_bgr, slice_size=320, overlap=0.2):
    res = get_sliced_prediction(img_bgr, sahi_model, slice_height=slice_size, slice_width=slice_size,
                                overlap_height_ratio=overlap, overlap_width_ratio=overlap, verbose=0)
    return res.object_prediction_list

# Compare on test images: normal vs SAHI detection counts
yolo = YOLO(best)
tests = sorted(glob.glob(str(CLAHE_DIR/'images/test/*.*')))[:30]
n_norm = n_sahi = 0
for p in tests:
    img = cv2.imread(p)
    n_norm += len(yolo.predict(img, conf=0.25, imgsz=IMGSZ, verbose=False)[0].boxes)
    n_sahi += len(detect_sahi(img))
print(f'Detections on 30 test images -> normal: {n_norm} | SAHI: {n_sahi}')

Detections on 30 test images -> normal: 84 | SAHI: 122


## 5. Tracking + track-level voting + unique counting + HUD video

In [7]:
VIDEO_IN  = 'thermal_input.mp4'      # upload a REAL thermal drone video (not stitched images)
VIDEO_OUT = 'thirdeye_output.mp4'
CONF, MIN_TRACK_FRAMES, SPEED_WINDOW = 0.25, 5, 10
COLORS = [(0,255,0),(0,128,255),(255,0,0),(0,255,255),(255,0,255),(255,255,0),(128,128,255)]

class TrackMemory:
    """Per-track class voting, trajectory and speed. Fixes flickering misclassification."""
    def __init__(self): self.votes=defaultdict(Counter); self.hist=defaultdict(list); self.seen=Counter()
    def update(self, tid, cls, conf, center, frame_idx):
        self.votes[tid][cls] += conf          # confidence-weighted vote
        self.hist[tid].append((frame_idx, center)); self.seen[tid] += 1
    def label(self, tid): return self.votes[tid].most_common(1)[0][0]
    def velocity(self, tid, fps):
        h = self.hist[tid][-SPEED_WINDOW:]
        if len(h) < 2: return 0.0, (0.0, 0.0)
        (f0,(x0,y0)), (f1,(x1,y1)) = h[0], h[-1]
        dt = max(f1-f0, 1)/fps
        vx, vy = (x1-x0)/dt, (y1-y0)/dt
        return float(np.hypot(vx, vy)), (vx, vy)      # pixels/sec (needs calibration for m/s)
    def counts(self):
        c = Counter()
        for tid, n in self.seen.items():
            if n >= MIN_TRACK_FRAMES: c[self.label(tid)] += 1   # unique targets, not per-frame boxes
        return c

def draw_hud(frame, mem, active):
    counts = mem.counts()
    h = 30 + 24*len(CLASS_NAMES)
    ov = frame.copy(); cv2.rectangle(ov, (8,8), (300,h), (0,0,0), -1)
    frame = cv2.addWeighted(ov, 0.55, frame, 0.45, 0)
    cv2.putText(frame, f'THIRDEYE | active: {active}', (16,28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
    for i, n in enumerate(CLASS_NAMES):
        cv2.putText(frame, f'{n}: {counts.get(i,0)}', (16, 54+24*i), cv2.FONT_HERSHEY_SIMPLEX, 0.55, COLORS[i%len(COLORS)], 1)
    return frame

def run_tracking(video_in, video_out, weights):
    model = YOLO(str(weights)); mem = TrackMemory()
    cap = cv2.VideoCapture(video_in)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W, H = int(cap.get(3)), int(cap.get(4))
    out = cv2.VideoWriter(video_out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    fi = 0
    while True:
        ok, raw = cap.read()
        if not ok: break
        frame = preprocess(raw)                                    # CLAHE
        r = model.track(frame, tracker='bytetrack.yaml', persist=True, conf=CONF,
                        imgsz=IMGSZ, verbose=False)[0]
        vis = frame.copy(); active = 0
        if r.boxes.id is not None:
            for box, tid, c, cf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(),
                                       r.boxes.cls.int().cpu().tolist(), r.boxes.conf.cpu().tolist()):
                x1,y1,x2,y2 = map(int, box); cx, cy = (x1+x2)//2, (y1+y2)//2
                mem.update(tid, c, cf, (cx,cy), fi); active += 1
                lab = mem.label(tid); col = COLORS[lab % len(COLORS)]
                spd, (vx,vy) = mem.velocity(tid, fps)
                cv2.rectangle(vis, (x1,y1), (x2,y2), col, 2)
                cv2.putText(vis, f'#{tid} {CLASS_NAMES[lab]} {spd:.0f}px/s', (x1, max(y1-6,12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)
                if spd > 5:   # direction arrow
                    n = max(np.hypot(vx,vy), 1e-6)
                    cv2.arrowedLine(vis, (cx,cy), (int(cx+vx/n*40), int(cy+vy/n*40)), col, 2, tipLength=0.3)
        out.write(draw_hud(vis, mem, active)); fi += 1
    cap.release(); out.release()
    print('Unique targets counted:', {CLASS_NAMES[k]: v for k, v in mem.counts().items()})
    return mem

mem = run_tracking(VIDEO_IN, VIDEO_OUT, w_clahe)

Unique targets counted: {}


## 6. (Future-scope demo) Data-scaling curve
Train on 25 / 50 / 100 % of the data and plot mAP. A rising curve is your evidence that **more data ⇒ better model**.

In [8]:
import matplotlib.pyplot as plt
if not RUN_SCALING: raise SystemExit('Scaling skipped')
all_train = sorted(glob.glob(str(CLAHE_DIR/'images/train/*.*')))
random.seed(0); random.shuffle(all_train)
SCALE_EPOCHS = 15; SCALE_IMGSZ = 640
scores = {}
for frac in [0.25, 0.5, 1.0]:
    sub = all_train[:int(len(all_train)*frac)]
    txt = Path(f'/content/train_{int(frac*100)}.txt'); txt.write_text('\n'.join(sub))
    cfg = dict(data_yaml); cfg.update(path=str(CLAHE_DIR), train=str(txt), val='images/val', test='images/test')
    y = Path(f'/content/scale_{int(frac*100)}.yaml'); yaml.safe_dump(cfg, open(y,'w'))
    m = YOLO(MODEL_BASE)
    m.train(data=str(y), epochs=SCALE_EPOCHS, imgsz=SCALE_IMGSZ, batch=BATCH, workers=4, device=0, project='ThirdEye',
            name=f'scale_{int(frac*100)}', exist_ok=True, **AUG)
    scores[frac*100] = m.val(data=str(y), split='test', imgsz=SCALE_IMGSZ, verbose=False).box.map50
plt.plot(list(scores), list(scores.values()), 'o-'); plt.xlabel('% of training data'); plt.ylabel('mAP50 (test)')
plt.title('Data scaling: more data -> better detector'); plt.grid(True); plt.savefig('scaling_curve.png', dpi=150); plt.show()

New https://pypi.org/project/ultralytics/8.4.187 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.75  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=\content\scale_25.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0

<Figure size 640x480 with 1 Axes>

## Future scope (for the presentation)
Active-learning data engine · RGB→thermal synthetic data (CycleGAN) · aircraft & fine-grained classes · RGB–thermal sensor fusion · vision-language alerts · TensorRT multi-stream deployment.